# Deployment-to-model-response benchmark

This notebook implements the prespecified deployment-readiness outcome:

> **Elapsed time from immediately before submission of the commit-pinned GitHub ARM template to the first valid model result returned by `POST /api/analyze`.**

The protocol comprises five randomized complete blocks. Each of the five models is evaluated once per block, yielding 25 serial attempts in Sweden Central. Failed attempts are retained without replacement. Preflight and cleanup are excluded from the measured interval. Cleanup is verified before each subsequent attempt; a completed campaign therefore leaves no benchmark resources.

Resource creation is disabled by default (`EXECUTE = False`).

## 1. Dependencies and configuration

Azure CLI performs deployment operations. Python packages provide tabular analysis and visualization.

In [1]:
%pip install -q requests pandas matplotlib

Note: you may need to restart the kernel to use updated packages.


In [12]:
from __future__ import annotations

import hashlib
import json
import platform
import random
import shutil
import subprocess
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "infra" / "main.json").exists():
    REPO_ROOT = REPO_ROOT.parent
NOTEBOOK = REPO_ROOT / "deployment-benchmark.ipynb"
RESULTS_ROOT = REPO_ROOT / "deployment-benchmark-results"

MEASUREMENT_SITE = "Berlin, Germany; wired hospital network"
EXECUTE = False  # Set to True only for the prespecified 25-attempt campaign.

REPOSITORY = "helloworld-germany/inconsistency-check"
MODELS = ("claude-opus-4-7", "gpt-5.5", "mistral-large-3", "deepseek-v3.2", "gpt-5.4-nano")
REPETITIONS, REGION, CAPACITY, SEED = 5, "swedencentral", 20, 20260901
TIMEOUT_S, POLL_S, REQUEST_TIMEOUT_S = 1800, 5, 190
PACKAGE_URI = f"https://github.com/{REPOSITORY}/releases/download/v1.0.0/app.zip"
PACKAGE_SHA256 = "83db273fa62a650fc30f1a9c2512a5ae351a25039cf59d2a4f61a3e3ae948b76"
PROBE = {"text": "Admission date: 2026-05-15. Discharge date: 2026-05-12. Discharge occurred three days after admission."}

random.seed(SEED)
plt.style.use("seaborn-v0_8-whitegrid")
display(pd.DataFrame({"component": ["Python", "pandas", "requests"],
                      "version": [platform.python_version(), pd.__version__, requests.__version__]}))

,component,version
0,Python,3.14.6
1,pandas,3.0.5
2,requests,2.34.2


## 2. Existing benchmark data

Each campaign writes checkpointed raw observations and metadata to a separate directory. When prior results are available, the following cell reports their structure, data types, and missing values.

In [3]:
prior_files = sorted(RESULTS_ROOT.glob("*/raw.csv")) if RESULTS_ROOT.exists() else []
if prior_files:
    prior = pd.read_csv(prior_files[-1])
    print(f"Latest results: {prior_files[-1]}")
    display(prior)
    display(pd.DataFrame({"dtype": prior.dtypes.astype(str), "missing": prior.isna().sum()}))
else:
    prior = pd.DataFrame()
    print("No previous benchmark results found.")

No previous benchmark results found.


## 3. Reproducible benchmark execution

The PHI-free probe contains one temporal contradiction and requires no domain-specific medical knowledge. Preflight verifies immutable inputs by URI and SHA-256, constructs the deterministic randomized schedule, and confirms Azure authentication. Each attempt ends at the first valid model response and is followed by verified cleanup.

In [4]:
def run(executable: str, *args: str, check: bool = True) -> subprocess.CompletedProcess[str]:
    result = subprocess.run([executable, *args], cwd=REPO_ROOT, text=True,
                            capture_output=True, encoding="utf-8", errors="replace")
    if check and result.returncode:
        message = (result.stderr or result.stdout).strip().replace("\n", " ")[:600]
        raise RuntimeError(message or f"Command failed: {args[0]}")
    return result


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def sha256(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


AZ, GIT = shutil.which("az"), shutil.which("git")
if not AZ or not GIT:
    raise RuntimeError("Azure CLI and Git must both be available on PATH.")

commit = run(GIT, "rev-parse", "HEAD").stdout.strip().lower()
if len(commit) != 40:
    raise RuntimeError("Could not resolve a 40-character Git commit.")
tracked_changes = run(GIT, "status", "--porcelain", "--untracked-files=no").stdout.strip()
notebook_in_commit = run(GIT, "cat-file", "-e", "HEAD:deployment-benchmark.ipynb", check=False).returncode == 0
if EXECUTE and (tracked_changes or not notebook_in_commit):
    raise RuntimeError("Commit and push the notebook and all tracked changes before execution.")

account = json.loads(run(AZ, "account", "show", "-o", "json", "--only-show-errors").stdout)
az_version = json.loads(run(AZ, "version", "-o", "json").stdout).get("azure-cli", "unknown")
template_uri = f"https://raw.githubusercontent.com/{REPOSITORY}/{commit}/infra/main.json"
template_bytes = requests.get(template_uri, timeout=120).content
package_response = requests.get(PACKAGE_URI, timeout=300)
package_response.raise_for_status()
package_bytes = package_response.content
template_sha256, package_sha256 = sha256(template_bytes), sha256(package_bytes)
if package_sha256 != PACKAGE_SHA256:
    raise RuntimeError(f"Package SHA-256 mismatch: {package_sha256}")
required = {"nameSuffix", "modelProfile", "modelCapacity", "packageUri"}
if not required <= set(json.loads(template_bytes)["parameters"]):
    raise RuntimeError("The pinned template does not expose the required parameters.")

rng, rows, sequence = random.Random(SEED), [], 0
for block in range(1, REPETITIONS + 1):
    order = list(MODELS)
    rng.shuffle(order)
    for model in order:
        sequence += 1
        rows.append({"sequence": sequence, "block": block, "model": model})
plan = pd.DataFrame(rows)

print(f"Preflight passed; Azure environment={account['environmentName']}; CLI={az_version}")
print(f"Pinned template: {template_uri}\nTemplate SHA-256: {template_sha256}\nPackage SHA-256:  {package_sha256}")
if not notebook_in_commit:
    print("Execution remains blocked until this notebook is committed and pushed.")
display(plan)

Preflight passed; Azure environment=AzureCloud; CLI=2.87.0
Pinned template: https://raw.githubusercontent.com/helloworld-germany/inconsistency-check/7858c520eb92fff254864ea2f5c884fb23d0a502/infra/main.json
Template SHA-256: 70a2da4bd6ffd2e6252283935ab6110245b5a09e6daff05dec9b3eda643b0850
Package SHA-256:  83db273fa62a650fc30f1a9c2512a5ae351a25039cf59d2a4f61a3e3ae948b76
Execution remains blocked until this notebook is committed and pushed.


,sequence,block,model
0,1,1,gpt-5.4-nano
1,2,1,gpt-5.5
2,3,1,mistral-large-3
3,4,1,claude-opus-4-7
4,5,1,deepseek-v3.2
5,6,2,deepseek-v3.2
6,7,2,gpt-5.5
7,8,2,claude-opus-4-7
8,9,2,gpt-5.4-nano
9,10,2,mistral-large-3


In [ ]:
TERMINAL_STATES = {"Succeeded", "Failed", "Canceled"}


def az(*args: str, check: bool = True) -> subprocess.CompletedProcess[str]:
    return run(AZ, *args, check=check)


def short_error(value: object) -> str:
    return str(value).strip().replace("\r", " ").replace("\n", " ")[:600]


def deployment_state(name: str) -> str | None:
    result = az("deployment", "sub", "show", "--name", name,
                "--query", "properties.provisioningState", "-o", "tsv",
                "--only-show-errors", check=False)
    return result.stdout.strip() if result.returncode == 0 else None


def group_exists(name: str) -> bool:
    return az("group", "exists", "--name", name, "-o", "tsv").stdout.strip().lower() == "true"


def deployment_exists(name: str) -> bool:
    names = json.loads(az("deployment", "sub", "list", "--query", "[].name", "-o", "json").stdout)
    return name in names


def purge_account(group: str, account: str) -> bool:
    for _ in range(60):
        result = az("cognitiveservices", "account", "purge", "--location", REGION,
                    "--resource-group", group, "--name", account,
                    "-o", "none", "--only-show-errors", check=False)
        if result.returncode == 0:
            return True
        time.sleep(5)
    return False


def cleanup(record: dict) -> None:
    group, account_name, deployment = record["resource_group"], f"aif-{record['suffix']}", record["deployment_name"]
    errors: list[str] = []
    state = deployment_state(deployment)
    if state and state not in TERMINAL_STATES:
        result = az("deployment", "sub", "cancel", "--name", deployment,
                    "--only-show-errors", check=False)
        if result.returncode:
            errors.append(f"cancel: {short_error(result.stderr)}")
        deadline = time.monotonic() + 300
        while time.monotonic() < deadline and (state := deployment_state(deployment)) not in TERMINAL_STATES:
            time.sleep(5)
        if state not in TERMINAL_STATES:
            errors.append(f"deployment remained {state or 'unknown'}")

    show = az("cognitiveservices", "account", "show", "-g", group, "-n", account_name,
              "-o", "none", "--only-show-errors", check=False)
    had_account, purged = show.returncode == 0, False
    if had_account:
        listed = az("cognitiveservices", "account", "deployment", "list", "-g", group,
                    "-n", account_name, "--query", "[].name", "-o", "tsv",
                    "--only-show-errors", check=False)
        if listed.returncode:
            errors.append(f"model list: {short_error(listed.stderr)}")
        else:
            for model_deployment in listed.stdout.splitlines():
                deleted = az("cognitiveservices", "account", "deployment", "delete",
                             "-g", group, "-n", account_name, "--deployment-name",
                             model_deployment, "--only-show-errors", check=False)
                if deleted.returncode:
                    errors.append(f"model delete: {short_error(deleted.stderr)}")
        deleted = az("cognitiveservices", "account", "delete", "-g", group,
                     "-n", account_name, "-o", "none",
                     "--only-show-errors", check=False)
        if deleted.returncode:
            errors.append(f"account delete: {short_error(deleted.stderr)}")
        else:
            purged = purge_account(group, account_name)

    if group_exists(group):
        deleted = az("group", "delete", "--name", group, "--yes", "-o", "none",
                     "--only-show-errors", check=False)
        if deleted.returncode:
            errors.append(f"group delete: {short_error(deleted.stderr)}")
    if had_account and not purged:  # Fallback after resource-group deletion.
        purged = purge_account(group, account_name)
    if had_account and not purged:
        errors.append("account purge could not be verified")

    if deployment_exists(deployment):
        deleted = az("deployment", "sub", "delete", "--name", deployment,
                     "--only-show-errors", check=False)
        if deleted.returncode:
            errors.append(f"deployment record: {short_error(deleted.stderr)}")
    if group_exists(group) or deployment_exists(deployment):
        errors.append("residual resource group or deployment record detected")
    if errors:
        raise RuntimeError(" | ".join(errors))


def wait_for_model(url: str, run_id: str, started: float) -> tuple[float | None, int, str | None]:
    attempts, last_error = 0, None
    while (elapsed := time.perf_counter() - started) < TIMEOUT_S:
        attempts += 1
        try:
            response = requests.post(f"{url}/api/analyze?benchmark={run_id}-{attempts}",
                                     json=PROBE, headers={"Cache-Control": "no-cache"},
                                     timeout=(5, min(REQUEST_TIMEOUT_S, max(1, TIMEOUT_S - elapsed))))
            response.raise_for_status()
            body = response.json()
            if isinstance(body, dict) and "issues" in body:
                return round(time.perf_counter() - started, 3), attempts, None
            last_error = "HTTP 200 JSON did not contain an issues property"
        except Exception as exc:
            last_error = short_error(exc)
        time.sleep(min(POLL_S, max(0, TIMEOUT_S - (time.perf_counter() - started))))
    return None, attempts, last_error

In [6]:
RUN_COLUMNS = ["sequence", "block", "model", "suffix", "resource_group", "deployment_name",
               "started_utc", "response_utc", "duration_s", "attempts", "outcome", "error",
               "cleanup", "cleanup_error"]


def save_checkpoint(report: dict, directory: Path) -> None:
    directory.mkdir(parents=True, exist_ok=True)
    json_tmp, csv_tmp = directory / "report.json.tmp", directory / "raw.csv.tmp"
    json_tmp.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
    pd.DataFrame(report["runs"], columns=RUN_COLUMNS).to_csv(csv_tmp, index=False)
    json_tmp.replace(directory / "report.json")
    csv_tmp.replace(directory / "raw.csv")


def new_record(spec: dict, campaign_id: str) -> dict:
    suffix = "b" + uuid.uuid4().hex[:7]
    return {**spec, "suffix": suffix, "resource_group": f"rg-logiccheck-{suffix}",
            "deployment_name": f"lc-{campaign_id}-{spec['sequence']}", "started_utc": None,
            "response_utc": None, "duration_s": None, "attempts": 0, "outcome": "pending",
            "error": None, "cleanup": "pending", "cleanup_error": None}


def measure(record: dict, campaign_id: str) -> None:
    record["started_utc"], started = utc_now(), time.perf_counter()
    try:
        submitted = az("deployment", "sub", "create", "--location", REGION,
                       "--name", record["deployment_name"], "--template-uri", template_uri,
                       "--parameters", f"nameSuffix={record['suffix']}",
                       f"modelProfile={record['model']}", f"modelCapacity={CAPACITY}",
                       f"packageUri={PACKAGE_URI}", "--no-wait", "-o", "none",
                       "--only-show-errors", check=False)
        if submitted.returncode:
            record.update(outcome="submission_failed", error=short_error(submitted.stderr or submitted.stdout))
        else:
            url = f"https://func-lc-{record['suffix']}.azurewebsites.net"
            duration, attempts, error = wait_for_model(url, f"{campaign_id}-{record['sequence']}", started)
            record.update(duration_s=duration, attempts=attempts, error=error,
                          outcome="succeeded" if duration is not None else "timed_out",
                          response_utc=utc_now() if duration is not None else None)
    except Exception as exc:
        record.update(outcome="script_error", error=short_error(exc))
    finally:
        try:
            cleanup(record)
            record["cleanup"] = "completed"
        except Exception as exc:
            record.update(cleanup="failed", cleanup_error=short_error(exc))


def residuals(records: list[dict]) -> list[str]:
    deployment_names = set(json.loads(az("deployment", "sub", "list", "--query", "[].name", "-o", "json").stdout))
    remaining = []
    for record in records:
        if group_exists(record["resource_group"]):
            remaining.append(record["resource_group"])
        if record["deployment_name"] in deployment_names:
            remaining.append(record["deployment_name"])
    return remaining

In [7]:
if not EXECUTE:
    print("Execution disabled. Preflight is safe; set EXECUTE = True only for the 25-run campaign.")
else:
    if run(GIT, "status", "--porcelain", "--untracked-files=no").stdout.strip() or not notebook_in_commit:
        raise RuntimeError("Execution requires a clean, committed and pushed protocol.")

    campaign_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:6]
    result_dir = RESULTS_ROOT / campaign_id
    metadata = {
        "protocol": "deployment-to-model-response-v1.0", "campaign_id": campaign_id,
        "state": "running", "started_utc": utc_now(), "completed_utc": None,
        "metric": "seconds from immutable ARM submission to first valid POST /api/analyze result",
        "result_contract": "HTTP 200 JSON containing an issues property",
        "design": "5 randomized complete blocks; serial; failures not replaced",
        "measurement_site": MEASUREMENT_SITE, "region": REGION, "model_capacity_k_tpm": CAPACITY,
        "seed": SEED, "timeout_s": TIMEOUT_S, "poll_s": POLL_S,
        "request_timeout_s": REQUEST_TIMEOUT_S, "probe": PROBE,
        "repository_commit": commit, "template_uri": template_uri,
        "template_sha256": template_sha256, "package_uri": PACKAGE_URI,
        "package_sha256": package_sha256,
        "notebook_sha256_at_start": sha256(NOTEBOOK.read_bytes()),
        "azure_environment": account["environmentName"], "azure_cli_version": az_version,
        "python_version": platform.python_version(), "platform": platform.platform(),
    }
    report = {"metadata": metadata, "plan": plan.to_dict("records"), "runs": []}
    save_checkpoint(report, result_dir)
    fatal_error = None

    try:
        for spec in report["plan"]:
            record = new_record(spec, campaign_id)
            report["runs"].append(record)       # Persist names before creating anything.
            save_checkpoint(report, result_dir)
            print(f"[{record['sequence']:02d}/25] block={record['block']} model={record['model']}")
            measure(record, campaign_id)
            save_checkpoint(report, result_dir)
            print(f"  outcome={record['outcome']} duration={record['duration_s']} s cleanup={record['cleanup']}")
            if record["cleanup"] != "completed" or record["outcome"] == "script_error":
                raise RuntimeError(record["cleanup_error"] or record["error"])
    except Exception as exc:
        fatal_error = short_error(exc)
    finally:
        leftovers = residuals(report["runs"])
        metadata.update(completed_utc=utc_now(),
                        state="aborted" if fatal_error or leftovers else "completed",
                        residual_resources=leftovers)
        save_checkpoint(report, result_dir)

    if leftovers:
        raise RuntimeError(f"Residual resources require recovery: {leftovers}")
    if fatal_error:
        raise RuntimeError(f"Campaign aborted: {fatal_error}")
    print(f"Campaign completed with no residual resources: {result_dir}")

Execution disabled. Preflight is safe; set EXECUTE = True only for the 25-run campaign.


In [8]:
RECOVER_LATEST = False
if RECOVER_LATEST:
    reports = sorted(RESULTS_ROOT.glob("*/report.json"))
    if not reports:
        raise RuntimeError("No checkpoint report found.")
    recovery_dir = reports[-1].parent
    recovery = json.loads(reports[-1].read_text(encoding="utf-8"))
    for record in recovery["runs"]:
        try:
            cleanup(record)
            record.update(cleanup="completed", cleanup_error=None)
        except Exception as exc:
            record.update(cleanup="failed", cleanup_error=short_error(exc))
    recovery["metadata"]["residual_resources"] = residuals(recovery["runs"])
    save_checkpoint(recovery, recovery_dir)
    if recovery["metadata"]["residual_resources"]:
        raise RuntimeError(recovery["metadata"]["residual_resources"])
    print("Recovery completed; no residual benchmark resources found.")

## 4. Descriptive analysis

The summary retains unsuccessful attempts in the denominator and reports the prespecified endpoint by model: successful and attempted counts, arithmetic mean, sample standard deviation, range, and individual observations. No hypothesis test is performed because only five attempts are planned per model.

In [9]:
reports = sorted(RESULTS_ROOT.glob("*/raw.csv")) if RESULTS_ROOT.exists() else []
if not reports:
    print("No measured observations yet; run the guarded campaign first.")
    results, summary = pd.DataFrame(), pd.DataFrame()
else:
    analysis_dir = reports[-1].parent
    results = pd.read_csv(reports[-1])
    rows = []
    for model in MODELS:
        records = results[results["model"] == model]
        values = records.loc[records["outcome"] == "succeeded", "duration_s"].dropna().astype(float)
        rows.append({"model": model, "successful": len(values), "attempted": len(records),
                     "failed_or_timed_out": len(records) - len(values),
                     "mean_s": values.mean(), "sample_sd_s": values.std(ddof=1),
                     "min_s": values.min(), "max_s": values.max(),
                     "observations_s": "; ".join(f"{value:.3f}" for value in values)})
    summary = pd.DataFrame(rows).round({"mean_s": 1, "sample_sd_s": 1, "min_s": 1, "max_s": 1})
    summary.to_csv(analysis_dir / "summary.csv", index=False)
    display(summary)

    MODEL_FILTER = "all"  # Replace with one model name to inspect individual attempts.
    display(results if MODEL_FILTER == "all" else results[results["model"] == MODEL_FILTER])

No measured observations yet; run the guarded campaign first.


## 5. Distribution visualization

Each point represents one successful deployment-to-response observation. The black marker and error bar denote the arithmetic mean and sample standard deviation. Unsuccessful attempts are not imputed and remain explicit in the reported counts.

In [10]:
if results.empty:
    print("No observations to plot.")
else:
    fig, ax = plt.subplots(figsize=(10, 5.5), constrained_layout=True)
    jitter = random.Random(SEED)
    for x, model in enumerate(MODELS):
        values = results.loc[(results["model"] == model) & (results["outcome"] == "succeeded"),
                             "duration_s"].dropna().astype(float)
        ax.scatter([x + jitter.uniform(-0.07, 0.07) for _ in values], values,
                   s=48, alpha=0.8, color="#0072B2", zorder=3)
        if len(values):
            ax.errorbar(x, values.mean(), yerr=values.std(ddof=1) if len(values) > 1 else 0,
                        fmt="D", color="black", capsize=5, label="Mean ± sample SD" if x == 0 else None)
    ax.set_xticks(range(len(MODELS)), MODELS, rotation=20, ha="right")
    ax.set_ylabel("Deployment to first model response (seconds)")
    ax.set_xlabel("Model (five attempts planned per model)")
    ax.legend(frameon=False)
    figure_path = analysis_dir / "deployment-to-model-response.png"
    fig.savefig(figure_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved: {figure_path}")

No observations to plot.


## 6. Reproducibility artifacts

The campaign directory contains checkpointed raw data in CSV and JSON formats, a descriptive summary, and a publication-quality figure. The following cell adds a machine-readable summary and a SHA-256 artifact manifest.

In [11]:
if results.empty:
    print("Nothing to export yet.")
else:
    report_data = json.loads((analysis_dir / "report.json").read_text(encoding="utf-8"))
    summary_records = summary.astype(object).where(pd.notna(summary), None).to_dict("records")
    (analysis_dir / "summary.json").write_text(
        json.dumps({"metadata": report_data["metadata"], "summary": summary_records},
                   indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    artifacts = []
    for path in sorted(analysis_dir.iterdir()):
        if path.is_file() and path.name != "artifact-manifest.json":
            artifacts.append({"file": path.name, "bytes": path.stat().st_size,
                              "sha256": sha256(path.read_bytes())})
    (analysis_dir / "artifact-manifest.json").write_text(
        json.dumps(artifacts, indent=2), encoding="utf-8")
    display(pd.DataFrame(artifacts))

Nothing to export yet.


### Interpretation and limitations

The outcome estimates automated infrastructure readiness from deployment submission to the first valid model response. It includes Function App startup, networking, managed identity authentication, RBAC propagation, model availability, and one inference. It excludes portal data entry and clinician usability.

Given five attempts per model, the analysis is descriptive and reports individual observations, arithmetic mean, sample standard deviation, and range. Serial execution and verified account purging prevent concurrent quota consumption by the benchmark. Normal completion leaves no benchmark resources. Following forced kernel or host termination, the recovery cell must be executed and Azure must be inspected before the campaign is interpreted.